# 18 Void definitions compared

Three ways of drawing voids in the same TNG300 box, side by side:

| Definition | Works on | What a void is |
|---|---|---|
| **T-web basins** (notebook 17) | the T-web void class (λ_th = 0.65, σ_eff = 1 h⁻¹ Mpc), from the dark-matter density | a basin of the distance to the nearest non-void voxel; basins less than Δ deeper than the pass to a neighbor are merged |
| **VIDE** (pyVIDE, a pure-Python re-implementation of VIDE/ZOBOV) | the subhalos at spacing λ | a watershed basin of the Voronoi density of the tracers: the union of the Voronoi cells of its member tracers |
| **VoidFinder** (VAST; Hoyle & Vogeley 2002) | the subhalos at spacing λ | a union of empty spheres: the largest sphere (maximal sphere) plus smaller spheres overlapping it by at least half their volume, grown among the "wall" tracers after isolated tracers are removed |

DeepVoid's predictions at the same spacing go through the T-web basins with the same settings, so the model sits next to the two tracer-based finders at equal sparsity.

The point here is to look: how the size distributions compare, and what each definition's voids look like on the same slice. No matching between definitions yet.

**Settings used.** Spacings λ = 3, 5, 10 h⁻¹ Mpc (the same mass-selected subhalo samples DeepVoid uses), real space, each finder with its published default cuts:

- **VIDE, default cuts:** no zone merging (merging threshold 1e-9, as VIDE's generated scripts), then VIDE's standard catalog: radius above the mean tracer separation, top-level voids only, central-density cut.
- **VIDE, no cuts:** every watershed basin pyVIDE finds (pyVIDE deletes nothing by default).
- **ZOBOV-style hierarchy:** zones merged at the classic ZOBOV threshold of 0.2 times the mean density, which gives a parent/child tree; shown as top-level voids only and as all levels.
- **VoidFinder defaults:** wall/field split at the 3rd-nearest-neighbor distance (mean + 1.5σ), 5 h⁻¹ Mpc hole grid, maximal spheres of at least 10 h⁻¹ Mpc, holes joining a void when at least half inside its maximal sphere.

**Voxels.** Every catalog is turned into a label volume on the T-web grid (512³ for TNG) so all of them can be measured and drawn the same way. A VIDE void is the union of its tracers' Voronoi cells, and a tracer's Voronoi cell is the set of voxels nearer to it than to any other tracer, so the label volume is exact up to the voxel size. A VoidFinder void is the union of its spheres.

**Two more measures** (sections 9 and 10):

- **Contamination:** what each catalog's void volume is, in the true T-web classes: how much of it is true void, wall, filament or halo, and how much of the true void volume it covers.
- **Stability with sparsity:** each method's catalog at λ = 5 and 10 matched to the same method's catalog at λ = 3 (strict and merge-aware matching, as notebook 17). This measures how each definition holds up as the tracers thin out, without scoring anyone against the T-web definition.

**Boxes.** `BOX = 'tng'` (TNG300, the box DeepVoid was trained on) or `'smdpl'` (SMDPL, 400 h⁻¹ Mpc, Rockstar halos, Planck 2013; never seen by the model). SMDPL uses the T-web labels and density grid from notebooks 15 and 16 (512³, 0.78 h⁻¹ Mpc); DeepVoid's SMDPL predictions are made here once on a GPU, as in notebook 16, and saved.

**Time estimates.** Loops show a progress bar with the time left. Every slow single step (a finder run, a distance transform, a set of basins, a matching) prints how long the same step took on the last run, or at another spacing if this one is new, and records its own time in `deepvoid_runs/voids/timings.json`.

**Runtime.** CPU, High-RAM for TNG once its predictions exist (notebook 17). SMDPL needs a GPU (L4, High-RAM) for its first run, to predict; reruns are CPU. Everything slow (the finders, the voxel volumes, the basins, the matches) is cached on Drive and read back on reruns.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!rm -rf /content/deepvoid
!git clone -q "https://github.com/skumagai3/deepvoid.git" /content/deepvoid
!pip -q install --force-reinstall --no-deps /content/deepvoid
!pip -q install pyvide==1.0.0 "git+https://github.com/DESI-UR/VAST.git@24e66e197bc88c924b42c338ac82f5b538d66dd3"
!git -C /content/deepvoid log --oneline -1

## 1 Settings

In [ ]:
import gc
import json
import os
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from scipy.spatial import cKDTree

from deepvoid import Catalog
from deepvoid import voids as V
from deepvoid.labels import VOID
from deepvoid.observe import select_by_spacing
from deepvoid.sources.tng import read_subhalos

RUNS = Path('/content/drive/MyDrive/deepvoid_runs')
OUT = RUNS / 'void_definitions'
OUT.mkdir(exist_ok=True)
PRED = RUNS / 'predictions'                          # whole-box scores (notebook 17 for TNG, made here for SMDPL)
PRED.mkdir(exist_ok=True)
catalog = Catalog()

BOX = 'tng'                                          # 'tng' or 'smdpl'
TNG_SUBHALOS = catalog.root / 'TNG' / 'TNG300-1_subhalos_snap99.npz'   # notebook 10
SMDPL = Path('/content/drive/MyDrive/data/SMDPL')
SMDPL_HALOS = SMDPL / '2026-11-15-22-18-55-728229.parquet'             # notebook 16
SMDPL_DENSITY = SMDPL / 'smdpl_dens512_z0.npy'                         # notebook 15
SMDPL_LABELS = SMDPL / 'smdpl_mask_tweb_th0.65_seff1.npz'              # notebook 16
LAMBDA_TH = 0.65
MODEL = 'subhalo_C_mixed_smooth2ch_depth5_distill'   # DeepVoid predictions (notebook 17); None to leave out
SPACINGS = [3, 5, 10]                                # h^-1 Mpc
PERSISTENCE = [0.5, 1.0, 2.0]                        # T-web basins: Δ in h^-1 Mpc
P_MAIN = 1.0                                         # Δ for DeepVoid and for the single-Δ comparisons
MIN_RADIUS = 1.0                                     # basins: drop voids whose largest empty sphere is smaller
MARGIN = 96                                          # padding (cells) for the periodic distance transform
R_CUT = 3.0                                          # h^-1 Mpc: voids counted have R_eff above this
COARSEN = 1                                          # kept at 1: shares notebook 17's cache
REFERENCE = 3                                        # section 10: each method's catalog at this λ is the reference

VIDE_MERGE = {'VIDE': 1e-9, 'ZOBOV': 0.2}            # merging thresholds (times the mean tracer density)
VF_SETTINGS = dict(hole_grid_edge_length=5.0, min_maximal_radius=10.0)   # VoidFinder defaults
NUM_CPUS = os.cpu_count()

USE_CACHE = True
REFRESH = False
CACHE_VERSION = 1           # notebook 17's cache: the TNG T-web basins are shared with it
CACHE = RUNS / 'voids' / 'cache'
CACHE.mkdir(parents=True, exist_ok=True)


TIMINGS = RUNS / 'voids' / 'timings.json'   # how long each slow step took on earlier runs (shared by 17 and 18)


def _kind(name):
    # the same step at another spacing or box counts as an estimate: 'L10_...' -> 'L*_...'
    import re
    return re.sub(r'(^|_)L\d+', r'\1L*', name)


def timed(name, fn):
    # Runs fn(), prints how long the same step took last time (an estimate), and records this run's time.
    t = json.loads(TIMINGS.read_text()) if TIMINGS.exists() else {}
    if name in t:
        print(f'{name}: last time {t[name] / 60:.1f} min', flush=True)
    elif _kind(name) in t:
        print(f'{name}: about {t[_kind(name)] / 60:.1f} min at another spacing last time', flush=True)
    else:
        print(f'{name}: no earlier timing', flush=True)
    t0 = time.time()
    res = fn()
    dt = time.time() - t0
    t = json.loads(TIMINGS.read_text()) if TIMINGS.exists() else {}
    t[name] = dt
    t[_kind(name)] = dt
    TIMINGS.write_text(json.dumps(t, indent=1))
    print(f'{name}: {dt / 60:.1f} min', flush=True)
    return res


def cached(name, fn):
    # As notebook 17: result of fn(), saved on Drive the first time and read back afterwards.
    path = CACHE / f'v{CACHE_VERSION}_{BOX}_th{LAMBDA_TH}_c{COARSEN}_{name}.npz'.replace(' ', '')
    if USE_CACHE and not REFRESH and path.exists():
        with np.load(path) as f:
            out = {k: f[k] for k in f.files}
        out = {k: v.astype(np.int32) if v.dtype == np.uint16 else v for k, v in out.items()}
        return out['_'] if list(out) == ['_'] else out
    res = timed(f'{BOX}_th{LAMBDA_TH}_{name}'.replace(' ', ''), fn)
    d = res if isinstance(res, dict) else {'_': res}
    d = {k: v.astype(np.uint16) if (np.issubdtype(np.asarray(v).dtype, np.integer) and np.asarray(v).ndim == 3
                                    and 0 <= v.min() and v.max() < 65535) else v for k, v in d.items()}
    if USE_CACHE:
        np.savez(path, **d)
    return res

## 2 Truth, predictions, tracers

The true T-web classes, DeepVoid's predicted void class at each spacing, and the tracer samples: the most massive subhalos (TNG300-1) or halos (SMDPL, Rockstar, hosts and subhalos) so that the mean separation is λ, the same selection as DeepVoid's inputs.

**Grid points.** A label volume's voxel i stands for the grid point at position (i + `GRID_SHIFT`) × cell. TNG's grids put point i at i × cell (the `grid.cic` convention), so `GRID_SHIFT = 0`. SMDPL's density table does not say; notebook 16 measured it by correlating with a halo grid, and the same check runs here. The tracer lookups and the sphere drawing use this, so that every catalog sits on the same points as the T-web labels.

**SMDPL predictions.** As notebook 16: halos gridded with CIC at 1024³ (0.39 h⁻¹ Mpc, the training cell), predicted, and averaged onto the 512³ label points. Needs a GPU; saved to `predictions/smdpl_<model>_L<λ>.npz` once.

In [ ]:
from deepvoid.grid import cic
from deepvoid.io import load_volume
from deepvoid.resample import block_mean, onto_coarse_points

if BOX == 'tng':
    truth_class = catalog.load('tng300-3/mask').data.astype(np.uint8)
    box = 205.0
    halos = read_subhalos(TNG_SUBHALOS, sim='TNG300-1')
    GRID_SHIFT = 0.0
elif BOX == 'smdpl':
    from deepvoid.sources.smdpl import BOX_SIZE, read_halos
    truth_class = load_volume(SMDPL_LABELS, dtype=None).data.astype(np.uint8)
    box = BOX_SIZE
    halos = read_halos(SMDPL_HALOS)
    # grid-point placement (notebook 16): which of the two placements of the 1024^3 halo grid on the 512^3
    # points correlates better with the density
    dens = np.load(SMDPL_DENSITY)
    logd = np.log1p(dens).ravel(); logd = (logd - logd.mean()) / logd.std()
    fine = cic(select_by_spacing(halos, 3), 2 * truth_class.shape[0]).data
    fine /= fine.mean()
    corr = {}
    for offset in (0, 1):
        h = np.log1p(onto_coarse_points(fine, offset)).ravel()
        corr[offset] = float(np.mean((h - h.mean()) / h.std() * logd))
    del fine, h, logd, dens
    OFFSET = max(corr, key=corr.get)
    GRID_SHIFT = 0.5 * OFFSET
    print('correlation with the density for each placement:', {k: round(v, 4) for k, v in corr.items()},
          f'-> point i at (i + {GRID_SHIFT}) cells')
else:
    raise ValueError(BOX)
truth_void = truth_class == VOID
n = truth_void.shape[0]
cell = box / n
print(f'{BOX}: {n}^3 cells of {cell:.3f} h^-1 Mpc; true void share {truth_void.mean():.3f}')

tracers = {s: select_by_spacing(halos, s).pos % box for s in SPACINGS}
for s, p in tracers.items():
    print(f'λ = {s}: {len(p):,} tracers')

pred_void = {}
model = None
for s in SPACINGS if MODEL else []:
    path = PRED / f'{BOX}_{MODEL}_L{s}.npz'
    if not path.exists() and BOX == 'smdpl':
        if model is None:
            from deepvoid.pretrained import Pretrained
            model = Pretrained.from_run(RUNS / MODEL, catalog=catalog)
        t0 = time.time()
        field = cic(select_by_spacing(halos, s), 2 * n).data
        probs = model.predict(field, box, float(s), out_dtype=np.float16)
        del field
        probs = onto_coarse_points(probs, OFFSET).astype(np.float16)
        np.savez(path, scores=probs, box_size=box, spacing=float(s), model=MODEL, offset=OFFSET)
        del probs
        gc.collect()
        print(f'λ = {s}: predicted in {time.time() - t0:.0f} s, saved {path.name}', flush=True)
    if not path.exists():
        print(f'λ = {s}: no saved prediction ({path.name}); for TNG run notebook 17 with this spacing in PREDICT')
        continue
    pred_void[s] = np.load(path)['scores'].argmax(-1) == VOID
    print(f'λ = {s}: predicted void share {pred_void[s].mean():.3f}, agreement with truth '
          f'{np.mean(pred_void[s] == truth_void):.3f}')
del halos, model
gc.collect()
G = f'g{GRID_SHIFT}'   # in cache names of everything placed on the grid from tracer positions

## 3 T-web basins

The truth at every Δ and DeepVoid's predictions at Δ = `P_MAIN`, with notebook 17's settings. Read from notebook 17's cache when it has them.

In [ ]:
dist = {}


def distance(which):
    void = truth_void if which == 'truth' else pred_void[which]
    tag = 'truth' if which == 'truth' else f'{MODEL}_L{which}'
    if which not in dist:
        dist[which] = cached(f'{tag}_distance', lambda: V.distance_to_structure(void, cell=cell, margin=MARGIN))
    return dist[which]


def basins(which, p):
    void = truth_void if which == 'truth' else pred_void[which]
    tag = 'truth' if which == 'truth' else f'{MODEL}_L{which}'
    return cached(f'{tag}_basins_p{p}_r{MIN_RADIUS}',
                  lambda: V.distance_voids(void, distance=distance(which), persistence=p, min_radius=MIN_RADIUS))

## 4 VIDE and the ZOBOV-style hierarchy

pyVIDE runs once per spacing and merging threshold; its files go to `void_definitions/vide/` and are read back on reruns. Then the catalogs:

- `VIDE, default cuts`: VIDE's standard ("trimmed") catalog, radius above the mean tracer separation, top-level, central-density cut;
- `VIDE, no cuts`: every basin;
- `ZOBOV 0.2, top level` and `ZOBOV 0.2, all levels`: the hierarchy from merging at 0.2, uncut.

For the label volumes each voxel takes the void of its nearest tracer. With nesting (the 0.2 hierarchy, all levels) a tracer is in several voids; the voxel takes the innermost.

In [ ]:
import pyvide
from tqdm.auto import tqdm

VIDE_DIR = OUT / 'vide'
VIDE_DIR.mkdir(exist_ok=True)


def run_vide(s, label):
    name = f'{BOX}_L{s}_{label}'
    if (VIDE_DIR / f'{name}_voids.npz').exists() and not REFRESH:
        return pyvide.load_catalog(str(VIDE_DIR), name)
    cat = timed(f'{BOX}_L{s}_pyVIDE_{label}', lambda: pyvide.identify_voids(
        str(VIDE_DIR), name, box, tracers[s], VIDE_mode='pyvide', minRadius=None,
        mergingThreshold=VIDE_MERGE[label], numDivisions=1, verbose=False, overwrite=True))
    print(f'λ = {s}, {label}: {cat.numVoids} basins', flush=True)
    return cat


def vide_catalogs(s):
    out = {}
    cat = run_vide(s, 'VIDE')
    out['VIDE, default cuts'] = pyvide.filter_catalog(cat, variant='all', minRadius=float(s), maxCentralDen=1e-9)
    out['VIDE, no cuts'] = pyvide.filter_catalog(cat, variant='untrimmed_all')
    cat = run_vide(s, 'ZOBOV')
    full = pyvide.filter_catalog(cat, variant='untrimmed_all')
    out['ZOBOV 0.2, all levels'] = full
    out['ZOBOV 0.2, top level'] = pyvide.filter_catalog(cat, variant='trimmed_nodencut_all')
    return out


def tracer_labels(cat, n_tracers):
    # Label of each tracer: row + 1 of the innermost void that holds it (0: in no void of this catalog).
    lab = np.zeros(n_tracers, np.int32)
    order = np.argsort(np.asarray(cat.treeLevel), kind='stable')   # outer first, inner overwrite
    for row in tqdm(order, desc='tracers to voids', leave=False):
        lab[pyvide.members(cat, int(cat.voidID[row]))] = row + 1
    return lab


def nearest_tracer(s):
    # Index of the nearest tracer to every voxel center (periodic): the Voronoi cells on the grid.
    def fn():
        tree = cKDTree(tracers[s], boxsize=box)
        idx = np.empty((n, n, n), np.int32)
        c = (np.arange(n) + GRID_SHIFT) * cell
        yy, zz = np.meshgrid(c, c, indexing='ij')
        for i in tqdm(range(n), desc=f'nearest tracer, λ = {s}', unit='plane'):
            pts = np.column_stack([np.full(n * n, c[i]), yy.ravel(), zz.ravel()])
            idx[i] = tree.query(pts, workers=-1)[1].reshape(n, n)
        return idx
    return cached(f'L{s}_nearest_tracer_{G}', fn)

## 5 VoidFinder

VAST VoidFinder with its defaults on each subhalo sample, in its periodic-box mode. Output in `void_definitions/voidfinder/`, read back on reruns. Its wall/field split uses a non-periodic neighbor search, so a few tracers near the box faces can be called isolated that would not be in a periodic search; they are left as VoidFinder treats them.

Sizes: R_eff from the volume of each void's own union of spheres (VoidFinder lets voids overlap a little, so each is measured on its own), and the radius of its maximal sphere. Volumes are measured on a grid of twice the T-web cell (0.8 h⁻¹ Mpc for TNG; the smallest sphere has radius 2.5), and the label volume uses every second grid point, which keeps the run short; the slice in section 8 is drawn at full resolution. Where two voids overlap, the drawing gives the voxel to the one with the larger maximal sphere.

In [ ]:
from astropy.io import fits
from vast.voidfinder import find_voids, wall_field_separation

VF_DIR = OUT / 'voidfinder'
VF_DIR.mkdir(exist_ok=True)


def run_voidfinder(s):
    name = f'{BOX}_L{s}_'
    path = VF_DIR / f'{name}VoidFinder_Output.fits'
    if not path.exists() or REFRESH:
        def run():
            wall, field = wall_field_separation(tracers[s].astype(np.float64), survey_name=name,
                                                out_directory=str(VF_DIR) + '/')
            find_voids(wall, name, str(VF_DIR) + '/', mask_type='periodic',
                       xyz_limits=np.array([[0.0, 0.0, 0.0], [box, box, box]]), num_cpus=NUM_CPUS, **VF_SETTINGS)
            return len(field)
        removed = timed(f'{BOX}_L{s}_VoidFinder', run)
        print(f'λ = {s}: VoidFinder removed {removed} isolated tracers', flush=True)
    with fits.open(path) as h:
        holes = {c: np.asarray(h['HOLES'].data[c], dtype=np.float64) for c in ('x', 'y', 'z', 'radius')}
        holes['void'] = np.asarray(h['HOLES'].data['void'], dtype=np.int64)
        maximal = {c: np.asarray(h['MAXIMALS'].data[c], dtype=np.float64) for c in ('x', 'y', 'z', 'radius')}
        maximal['void'] = np.asarray(h['MAXIMALS'].data['void'], dtype=np.int64)
    return holes, maximal


VF_CELL = 2 * cell          # grid for VoidFinder volumes: spheres have radius ≥ 2.5 h^-1 Mpc, so 2 cells is enough


def union_radii(holes):
    # R_eff of each void from the volume of its own union of spheres, measured on a grid of VF_CELL. Voids may
    # overlap each other a little (VoidFinder allows it), so each void is measured on its own.
    cell = VF_CELL
    out = []
    centers = np.column_stack([holes['x'], holes['y'], holes['z']])
    for v in tqdm(np.unique(holes['void']), desc='VoidFinder volumes', unit='void', leave=False):
        sel = np.flatnonzero(holes['void'] == v)
        c = centers[sel]
        c = c[0] + (c - c[0] + box / 2) % box - box / 2          # nearest periodic images of the first sphere
        r = holes['radius'][sel]
        lo = np.floor((c - r[:, None]).min(0) / cell).astype(int)
        hi = np.ceil((c + r[:, None]).max(0) / cell).astype(int)
        g = [(np.arange(lo[a], hi[a] + 1) + 0.5) * cell for a in range(3)]
        inside = np.zeros((len(g[0]), len(g[1]), len(g[2])), bool)
        for ci, ri in zip(c, r):
            inside |= ((g[0][:, None, None] - ci[0]) ** 2 + (g[1][None, :, None] - ci[1]) ** 2
                       + (g[2][None, None, :] - ci[2]) ** 2) <= ri * ri
        out.append(np.cbrt(3 * inside.sum() * cell ** 3 / (4 * np.pi)))
    return np.array(out)


VF_STEP = 2                 # VoidFinder label volumes on every 2nd grid point (the label grid's points 0, 2, 4 ...)


def sphere_labels(holes):
    # Union of each void's spheres on every VF_STEP-th grid point (periodic), for the share of the box in voids
    # and the comparisons; a point in two voids keeps the void with the larger maximal sphere (VoidFinder's order).
    m, step = n // VF_STEP, VF_STEP * cell
    lab = np.zeros((m, m, m), np.int32)
    centers = np.column_stack([holes['x'], holes['y'], holes['z']]) % box
    order = np.argsort(holes['void'], kind='stable')
    for k in tqdm(order, desc='VoidFinder spheres', unit='sphere', leave=False):
        c, r = centers[k], holes['radius'][k]
        u = (c - GRID_SHIFT * cell) / step                  # position in coarse-index units
        lo = np.floor(u - r / step).astype(int)
        hi = np.ceil(u + r / step).astype(int)
        ax = [np.arange(lo[a], hi[a] + 1) for a in range(3)]
        n_ = m
        gx, gy, gz = [(a * step + GRID_SHIFT * cell - c[i]) for i, a in enumerate(ax)]
        inside = (gx[:, None, None] ** 2 + gy[None, :, None] ** 2 + gz[None, None, :] ** 2) <= r * r
        ix = np.ix_(ax[0] % n_, ax[1] % n_, ax[2] % n_)
        block = lab[ix]
        block[inside & (block == 0)] = holes['void'][k] + 1
        lab[ix] = block
    return lab


def sphere_slice(holes, x0):
    # The same on the full-resolution slice at x = x0 only: each sphere cut by the plane is a disk.
    lab = np.zeros((n, n), np.int32)
    centers = np.column_stack([holes['x'], holes['y'], holes['z']]) % box
    dx = (centers[:, 0] - x0 + box / 2) % box - box / 2
    cut = np.flatnonzero(np.abs(dx) < holes['radius'])
    for k in cut[np.argsort(holes['void'][cut], kind='stable')]:
        c, r = centers[k, 1:], np.sqrt(holes['radius'][k] ** 2 - dx[k] ** 2)
        ax = [np.arange(np.floor((c[a] - r) / cell - GRID_SHIFT), np.ceil((c[a] + r) / cell - GRID_SHIFT) + 1).astype(int)
              for a in range(2)]
        inside = (((ax[0] + GRID_SHIFT) * cell - c[0])[:, None] ** 2
                  + ((ax[1] + GRID_SHIFT) * cell - c[1])[None, :] ** 2) <= r * r
        ix = np.ix_(ax[0] % n, ax[1] % n)
        block = lab[ix]
        block[inside & (block == 0)] = holes['void'][k] + 1
        lab[ix] = block
    return lab

## 6 All catalogs as label volumes

For each spacing: the label volume of every catalog (cached), the effective radius of each void from its voxel volume, the share of the box in voids, and one slice for section 8. Catalog sizes use each finder's own radius where it has one (VIDE's `radius`, from the Voronoi volume) and the voxel volume otherwise.

In [ ]:
SLICE = n // 2
X0 = (SLICE + GRID_SHIFT) * cell     # position of the slice
results = {}          # (catalog, λ or 'truth') -> summary row
LABELS = {}           # (catalog, λ or 'truth') -> function that returns its label volume (cached on Drive)
radii = {}            # (catalog, λ or 'truth') -> R_eff of every void, h^-1 Mpc
slices = {}           # (catalog, λ or 'truth') -> labels on the slice


def record(name, s, lab, r=None, extra=None, plane=None):
    vcell = box / lab.shape[0]
    counts = np.bincount(lab.ravel())
    vol = counts[1:] * vcell ** 3
    r_vox = np.cbrt(3 * vol[vol > 0] / (4 * np.pi))
    r = r_vox if r is None else np.asarray(r, dtype=float)
    radii[(name, s)] = r
    big = r > R_CUT
    results[(name, s)] = {'voids': int(big.sum()),
                          'median R_eff': float(np.median(r[big])) if big.any() else np.nan,
                          'largest R_eff': float(r.max()) if len(r) else np.nan,
                          'void share of box': float((lab > 0).mean()), **(extra or {})}
    slices[(name, s)] = lab[SLICE].copy() if plane is None else plane


for p in PERSISTENCE:
    LABELS[(f'T-web basins, Δ = {p}', 'truth')] = lambda p=p: basins('truth', p)
    record(f'T-web basins, Δ = {p}', 'truth', basins('truth', p))
dist.pop('truth', None)
gc.collect()

for s in SPACINGS:
    t0 = time.time()
    if s in pred_void:
        LABELS[(f'DeepVoid basins, Δ = {P_MAIN}', s)] = lambda s=s: basins(s, P_MAIN)
        record(f'DeepVoid basins, Δ = {P_MAIN}', s, basins(s, P_MAIN))
        dist.pop(s, None)
    nearest = nearest_tracer(s)
    for name, cat in vide_catalogs(s).items():
        tl = tracer_labels(cat, len(tracers[s]))
        key = f'L{s}_{name}_labels_{G}'.replace(',', '').replace(' ', '_')
        lab = cached(key, lambda: tl[nearest])
        LABELS[(name, s)] = lambda key=key, s=s, name=name: cached(
            key, lambda: tracer_labels(vide_catalogs(s)[name], len(tracers[s]))[nearest_tracer(s)])
        record(name, s, lab, r=np.asarray(cat.radius))
        del lab
    del nearest
    holes, maximal = run_voidfinder(s)
    key = f'L{s}_VoidFinder_labels_step{VF_STEP}_{G}'
    lab = cached(key, lambda: sphere_labels(holes))
    LABELS[('VoidFinder', s)] = lambda key=key, s=s: cached(key, lambda: sphere_labels(run_voidfinder(s)[0]))
    record('VoidFinder', s, lab, r=cached(f'L{s}_VoidFinder_union_radii', lambda: union_radii(holes)),
           plane=sphere_slice(holes, X0))
    radii[('VoidFinder, maximal sphere', s)] = maximal['radius']
    del lab
    gc.collect()
    print(f'λ = {s}: done in {time.time() - t0:.0f} s', flush=True)

table = pd.DataFrame({f'{k[0]} | {"truth" if k[1] == "truth" else "λ = " + str(k[1])}': v
                      for k, v in results.items()}).T
print(table.round(3).to_string())
(OUT / f'{BOX}_summary.json').write_text(json.dumps({f'{k[0]}|{k[1]}': v for k, v in results.items()}, indent=1))
np.savez(OUT / f'{BOX}_radii.npz', **{f'{k[0]}|{k[1]}': v for k, v in radii.items()})

## 7 Size distributions

Cumulative number density of voids larger than R, one row per spacing. Left: the watershed definitions (T-web basins of the truth at each Δ, DeepVoid's basins, VIDE, the ZOBOV-style hierarchy). Right: the sphere-based one (VoidFinder, by effective radius and by maximal-sphere radius), with the truth and DeepVoid at Δ = `P_MAIN` for reference. The dotted vertical line is 2λ, below which voids found from tracers are usually cut (Ronconi et al. 2019 cut at twice the mean tracer separation).

The radii are not the same quantity in every catalog: VIDE's radius is the effective radius of the void's Voronoi volume, VoidFinder's maximal-sphere radius is the largest empty sphere, the others are the effective radius of the void's voxels.

In [ ]:
STYLE = {
    'T-web basins, Δ = 0.5': dict(color='k', ls=':'),
    'T-web basins, Δ = 1.0': dict(color='k', ls='-'),
    'T-web basins, Δ = 2.0': dict(color='k', ls='--'),
    f'DeepVoid basins, Δ = {P_MAIN}': dict(color='#dd6b20', ls='-'),
    'VIDE, default cuts': dict(color='#2b6cb0', ls='-'),
    'VIDE, no cuts': dict(color='#2b6cb0', ls='--'),
    'ZOBOV 0.2, top level': dict(color='#38a169', ls='-'),
    'ZOBOV 0.2, all levels': dict(color='#38a169', ls='--'),
    'VoidFinder': dict(color='#6b46c1', ls='-'),
    'VoidFinder, maximal sphere': dict(color='#6b46c1', ls='--'),
}
LEFT = [k for k in STYLE if not k.startswith('VoidFinder')]
RIGHT = [f'T-web basins, Δ = {P_MAIN}', f'DeepVoid basins, Δ = {P_MAIN}', 'VoidFinder', 'VoidFinder, maximal sphere']
r_top = max(float(np.max(r)) for r in radii.values() if len(r)) * 1.05
bins = np.logspace(np.log10(R_CUT), np.log10(r_top), 80)
ticks = [t for t in (3, 5, 10, 20, 50, 100, 200) if t <= r_top]


def label_of(name):
    return name + (' (truth)' if name.startswith('T-web') else '')


fig, axes = plt.subplots(len(SPACINGS), 2, figsize=(13, 4.0 * len(SPACINGS)), squeeze=False)
for row, s in zip(axes, SPACINGS):
    for ax, names in zip(row, (LEFT, RIGHT)):
        for name in names:
            key = (name, 'truth') if name.startswith('T-web') else (name, s)
            if key in radii and len(radii[key]):
                ax.step(bins, V.size_function(radii[key], box, bins), where='post', lw=1.4,
                        label=label_of(name), **STYLE[name])
        ax.axvline(2 * s, color='0.4', ls=':', lw=1.2, label='2λ')
        ax.set_xscale('log'); ax.set_yscale('log')
        ax.set_xticks(ticks, [str(t) for t in ticks]); ax.minorticks_off()
        ax.set_xlabel(r'void radius $R$ ($h^{-1}$ Mpc; see text for each catalog)')
        ax.set_ylabel(r'$n(>R)$ ($h^3$ Mpc$^{-3}$)')
        ax.set_title(f'λ = {s} h⁻¹ Mpc: ' + ('watershed definitions' if names is LEFT else 'sphere-based definition'),
                     fontsize=10)
        ax.grid(alpha=0.3)
for col, x in ((0, 0.27), (1, 0.76)):
    h, l = [], []
    for ax in axes[:, col]:
        for hh, ll in zip(*ax.get_legend_handles_labels()):
            if ll not in l:
                h.append(hh); l.append(ll)
    fig.legend(h, l, loc='upper center', bbox_to_anchor=(x, 0.0), ncol=2, fontsize=8, frameon=False)
fig.suptitle(f'{BOX}: void size functions of each definition (voids with R > {R_CUT:g} h⁻¹ Mpc; '
             f'T-web truth at λ_th = {LAMBDA_TH}, the same in every row)')
fig.tight_layout(rect=(0, 0, 1, 0.97))
fig.savefig(OUT / f'{BOX}_size_functions.png', dpi=150, bbox_inches='tight')
plt.show()

## 8 What the voids look like

The same slice (x = middle of the box) for every definition, one row per spacing: the dark-matter density in gray, each void filled with its own random color (colors are not matched between panels), and the tracers within ±λ/2 of the slice as black dots. White with only the density showing is outside every void of that catalog. The T-web truth does not depend on λ; it is repeated in each row for comparison.

In [ ]:
SHOW = [f'T-web basins, Δ = {P_MAIN}', f'DeepVoid basins, Δ = {P_MAIN}', 'VIDE, default cuts', 'VIDE, no cuts',
        'ZOBOV 0.2, top level', 'VoidFinder']
DENSITY_ALPHA = 0.5
VOID_ALPHA = 0.6
if BOX == 'tng':
    dm = catalog.load('tng300-3/dm').data
    k = dm.shape[0] // n
    rho = dm[SLICE * k:(SLICE + 1) * k].mean(0)
    rho = block_mean(rho, k) if k > 1 else rho
else:
    dm = np.load(SMDPL_DENSITY, mmap_mode='r')
    rho = np.asarray(dm[SLICE], dtype=np.float32)
del dm
log_rho = np.log10(np.maximum(rho / rho.mean(), 1e-2))
rng = np.random.default_rng(0)
ext = [0, box, 0, box]

fig, axes = plt.subplots(len(SPACINGS), len(SHOW), figsize=(4.2 * len(SHOW), 4.4 * len(SPACINGS)), squeeze=False)
for row, s in zip(axes, SPACINGS):
    p = tracers[s]
    dx = (p[:, 0] - X0 + box / 2) % box - box / 2
    near = np.abs(dx) <= s / 2
    for ax, name in zip(row, SHOW):
        key = (name, 'truth') if name.startswith('T-web') else (name, s)
        ax.imshow(log_rho.T, origin='lower', extent=ext, cmap='gray_r', vmin=-1, vmax=1.5, alpha=DENSITY_ALPHA)
        if key in slices:
            sl = slices[key]
            colors = np.zeros((int(sl.max()) + 1, 4))
            colors[1:, :3] = rng.random((len(colors) - 1, 3)) * 0.75 + 0.25
            colors[1:, 3] = VOID_ALPHA
            ax.imshow(colors[sl].transpose(1, 0, 2), origin='lower', extent=ext, interpolation='nearest')
            share = results[key]['void share of box']
            ax.set_title(f'{name}\nλ = {s}' + ('' if name.startswith('T-web') else '')
                         + f'; {share:.0%} of the box in voids', fontsize=9)
        else:
            ax.set_title(f'{name}, λ = {s}: not available', fontsize=9)
        ax.scatter(p[near, 1], p[near, 2], s=1.5, c='k', lw=0)
        ax.set_xlim(0, box); ax.set_ylim(0, box)
        ax.set_xlabel('y (h⁻¹ Mpc)'); ax.set_ylabel('z (h⁻¹ Mpc)')
handles = [Line2D([], [], ls='', marker='o', ms=3, color='k', label='tracer within ±λ/2 of the slice'),
           Patch(facecolor='white', edgecolor='0.6', label='white/gray only: not in any void of that catalog '
                 '(gray = dark-matter density, log ρ/ρ̄ from −1 to 1.5)'),
           Patch(facecolor='#c08080', alpha=VOID_ALPHA, label='colored: one void each, random colors, not matched between panels')]
fig.suptitle(f'{BOX}: slice x = {X0:.0f} h⁻¹ Mpc, voids of each definition '
             f'(T-web truth at λ_th = {LAMBDA_TH}, same in every row)')
fig.tight_layout(rect=(0, 0, 1, 0.98), h_pad=3.0)
fig.legend(handles=handles, loc='upper center', bbox_to_anchor=(0.5, 0.0), ncol=3, fontsize=9, frameon=False)
fig.savefig(OUT / f'{BOX}_slices.png', dpi=130, bbox_inches='tight')
plt.show()
print('saved in', OUT)

## 9 Contamination: what each catalog's voids are made of

For every catalog, its void volume split by the true T-web class at the same grid points (void, wall, filament, halo), and the share of the true void volume that falls inside its voids (coverage). A definition whose voids follow the T-web voids has a high void share and a high coverage; one that draws voids across walls and filaments has a large wall and filament share. VoidFinder's label volume is on every second grid point, so it is compared with the truth on those points.

This uses the T-web classes as the yardstick, so it is not neutral between definitions: VIDE and VoidFinder never claimed to find T-web voids. It answers a narrower question: how much dark-matter structure lies inside the voids each one draws.

In [ ]:
CLASS_NAMES = ['void', 'wall', 'filament', 'halo']
contam = {}
for (name, s), get in LABELS.items():
    if name.startswith('T-web') and name != f'T-web basins, Δ = {P_MAIN}':
        continue
    lab = get()
    step = n // lab.shape[0]
    tc = truth_class[::step, ::step, ::step]
    inside = lab > 0
    share = np.bincount(tc[inside].ravel(), minlength=4) / max(inside.sum(), 1)
    coverage = float(inside[tc == VOID].mean())
    contam[(name, s)] = {**{f'{c} share': float(v) for c, v in zip(CLASS_NAMES, share)},
                         'coverage of true void': coverage, 'void share of box': float(inside.mean())}
    del lab, inside, tc
    gc.collect()
ctab = pd.DataFrame({f'{k[0]} | {"truth" if k[1] == "truth" else "λ = " + str(k[1])}': v
                     for k, v in contam.items()}).T
print(ctab.round(3).to_string())
(OUT / f'{BOX}_contamination.json').write_text(json.dumps({f'{k[0]}|{k[1]}': v for k, v in contam.items()},
                                                          indent=1))

SHORT = {f'T-web basins, Δ = {P_MAIN}': 'T-web\nbasins\n(truth)', f'DeepVoid basins, Δ = {P_MAIN}': 'DeepVoid\nbasins',
         'VIDE, default cuts': 'VIDE\ndefault\ncuts', 'VIDE, no cuts': 'VIDE\nno cuts',
         'ZOBOV 0.2, top level': 'ZOBOV 0.2\ntop level', 'ZOBOV 0.2, all levels': 'ZOBOV 0.2\nall levels',
         'VoidFinder': 'VoidFinder', 'DeepVoid vs T-web truth': 'DeepVoid\nvs T-web\ntruth'}
ORDER = [f'DeepVoid basins, Δ = {P_MAIN}', 'VIDE, default cuts', 'VIDE, no cuts', 'ZOBOV 0.2, top level',
         'ZOBOV 0.2, all levels', 'VoidFinder']
CLASS_COLORS = ['#cfe2f3', '#f6b26b', '#cc4125', '#5b0f00']
fig, axes = plt.subplots(1, len(SPACINGS), figsize=(5.2 * len(SPACINGS), 5.2), sharey=True, squeeze=False)
for ax, s in zip(axes[0], SPACINGS):
    names = [f'T-web basins, Δ = {P_MAIN}'] + [m for m in ORDER if (m, s) in contam]
    keys = [(m, 'truth') if m.startswith('T-web') else (m, s) for m in names]
    x = np.arange(len(keys))
    bottom = np.zeros(len(keys))
    for c, col in zip(CLASS_NAMES, CLASS_COLORS):
        v = np.array([contam[k][f'{c} share'] for k in keys])
        ax.bar(x, v, bottom=bottom, color=col, edgecolor='k', lw=0.4, label=f'true {c}' if s == SPACINGS[0] else None)
        bottom += v
    ax.plot(x, [contam[k]['coverage of true void'] for k in keys], 'kD', ms=6,
            label='coverage of the true void volume' if s == SPACINGS[0] else None)
    ax.set_xticks(x, [SHORT.get(m, m) for m in names], fontsize=8, rotation=0)
    ax.set_ylim(0, 1.02)
    ax.set_title(f'λ = {s} h⁻¹ Mpc')
    ax.grid(alpha=0.3, axis='y')
axes[0, 0].set_ylabel('share of the catalog\'s void volume (bars); coverage (diamonds)')
fig.legend(loc='upper center', bbox_to_anchor=(0.5, 0.0), ncol=5, fontsize=9, frameon=False)
fig.suptitle(f'{BOX}: what each catalog\'s voids are, in the true T-web classes (λ_th = {LAMBDA_TH})')
fig.tight_layout(rect=(0, 0, 1, 0.96))
fig.savefig(OUT / f'{BOX}_contamination.png', dpi=150, bbox_inches='tight')
plt.show()

## 10 Stability with sparsity: each method against itself

Each method's catalog at λ = 5 and 10 matched to the **same method's** catalog at λ = `REFERENCE` (3 h⁻¹ Mpc), the densest sample here, which plays the role of the truth. Matching as notebook 17: strict (Jaccard index > 0.5), and merge-aware (a sparse-sample void against the union of the reference voids matched to it or merged into it). Voids with R_eff > `R_CUT` counted. A method whose voids survive thinning the tracers has high completeness and purity here, whatever its definition.

Read merge-aware numbers together with `reference voids per matched void` in the table: a catalog with one box-filling void (the ZOBOV-style top level does this) is merge-aware "complete" trivially, because that void is a fair outline of the union of everything.

For DeepVoid the comparison with the T-web truth (notebook 17) is shown too, as a scale: at λ = 3 DeepVoid already reproduces the truth closely, so the two should be similar.

In [ ]:
FATES = ('matched', 'merged', 'split or shrunk', 'missed')


def fates(lab_t, lab_p, tp):
    # As notebook 17: 0 matched (Jaccard > 0.5); 1 merged (half or more of it in one larger void of the other
    # catalog); 2 split or shrunk; 3 missed (less than half of it is void in the other catalog).
    size_t = np.bincount(lab_t.ravel())[1:]
    size_p = np.bincount(lab_p.ravel(), minlength=int(lab_p.max()) + 1)
    in_void = np.bincount(lab_t.ravel(), weights=(lab_p > 0).ravel(), minlength=len(size_t) + 1)[1:]
    contain = tp['shared'] / np.maximum(size_t, 1)
    out = np.full(len(size_t), 2, np.int8)
    out[in_void < 0.5 * size_t] = 3
    out[(contain >= 0.5) & (size_p[tp['best']] > size_t)] = 1
    out[tp['jaccard'] > 0.5] = 0
    return out


def merge_aware(lab_t, lab_p, tp, fate):
    size_t = np.bincount(lab_t.ravel())[1:]
    n_p = int(lab_p.max())
    size_p = np.bincount(lab_p.ravel(), minlength=n_p + 1)[1:]
    assigned = (fate <= 1) & (tp['best'] > 0)
    j = tp['best'][assigned] - 1
    inter = np.bincount(j, weights=tp['shared'][assigned], minlength=n_p)
    union_t = np.bincount(j, weights=size_t[assigned], minlength=n_p)
    group = np.bincount(j, minlength=n_p)
    with np.errstate(invalid='ignore', divide='ignore'):
        jac = np.where(group > 0, inter / (size_p + union_t - inter), 0.0)
    ok_p = jac > 0.5
    ok_t = np.zeros(len(size_t), bool)
    ok_t[assigned] = ok_p[j]
    return ok_t, ok_p, group


def compare(ref_key, key):
    lab_t, lab_p = LABELS[ref_key](), LABELS[key]()
    step = n // lab_t.shape[0]
    vol_cell = (step * cell) ** 3
    r_t = np.cbrt(3 * np.bincount(lab_t.ravel())[1:] * vol_cell / (4 * np.pi))
    r_p = np.cbrt(3 * np.bincount(lab_p.ravel(), minlength=int(lab_p.max()) + 1)[1:] * vol_cell / (4 * np.pi))
    name = f'{BOX}_stab_{ref_key[0]}_{ref_key[1]}_vs_{key[0]}_{key[1]}_{G}'
    mm = cached(name.replace(',', '').replace('=', '').replace('Δ', 'D').replace(' ', '_'),
                lambda: {f'{a}_{k}': v for a, m in (('tp', V.match_voids(lab_t, lab_p)),
                                                    ('pt', V.match_voids(lab_p, lab_t))) for k, v in m.items()})
    tp = {k: mm[f'tp_{k}'] for k in ('best', 'shared', 'jaccard')}
    pt = {k: mm[f'pt_{k}'] for k in ('best', 'shared', 'jaccard')}
    fate = fates(lab_t, lab_p, tp)
    ok_t, ok_p, group = merge_aware(lab_t, lab_p, tp, fate)
    del lab_t, lab_p
    big_t, big_p = r_t > R_CUT, r_p > R_CUT
    row = {'reference voids': int(big_t.sum()), 'voids': int(big_p.sum()),
           'completeness': float(np.mean(tp['jaccard'][big_t] > 0.5)),
           'purity': float(np.mean(pt['jaccard'][big_p] > 0.5)),
           'completeness, merge-aware': float(ok_t[big_t].mean()),
           'purity, merge-aware': float(ok_p[big_p].mean()),
           'reference voids per matched void': float(group[ok_p & big_p & (group > 0)].mean())
           if (ok_p & big_p & (group > 0)).any() else np.nan}
    for i, f in enumerate(FATES[1:], start=1):
        row[f] = float(np.mean(fate[big_t] == i))
    return row


METHODS = [f'DeepVoid basins, Δ = {P_MAIN}', 'VIDE, default cuts', 'VIDE, no cuts', 'ZOBOV 0.2, top level',
           'VoidFinder']
stab = {}
for m in METHODS:
    if (m, REFERENCE) not in LABELS:
        continue
    for s in SPACINGS:
        if s == REFERENCE or (m, s) not in LABELS:
            continue
        t0 = time.time()
        stab[(m, s)] = compare((m, REFERENCE), (m, s))
        print(f'{m}, λ = {s} against λ = {REFERENCE}: {time.time() - t0:.0f} s', flush=True)
for s in SPACINGS:                                   # scale: DeepVoid against the T-web truth
    key = (f'DeepVoid basins, Δ = {P_MAIN}', s)
    if key in LABELS:
        stab[('DeepVoid vs T-web truth', s)] = compare((f'T-web basins, Δ = {P_MAIN}', 'truth'), key)
gc.collect()
stab_tab = pd.DataFrame({f'{k[0]} | λ = {k[1]}': v for k, v in stab.items()}).T
print(stab_tab.round(2).to_string())
(OUT / f'{BOX}_stability.json').write_text(json.dumps({f'{k[0]}|{k[1]}': v for k, v in stab.items()}, indent=1))

shown = [s for s in SPACINGS if s != REFERENCE]
methods = [m for m in METHODS + ['DeepVoid vs T-web truth'] if any((m, s) in stab for s in shown)]
fig, axes = plt.subplots(1, 2, figsize=(15, 5.2), sharey=True)
width = 0.8 / (2 * len(shown))
colors = {s: c for s, c in zip(shown, ['#2b6cb0', '#dd6b20', '#38a169'])}
for ax, what in zip(axes, ('completeness', 'purity')):
    x = np.arange(len(methods))
    for i, s in enumerate(shown):
        for j, mode in enumerate(('', ', merge-aware')):
            v = [stab.get((m, s), {}).get(what + mode, np.nan) for m in methods]
            ax.bar(x + (2 * i + j - len(shown) + 0.5) * width, v, width, color=colors[s],
                   alpha=1.0 if mode else 0.45, edgecolor='k', lw=0.4,
                   label=f'λ = {s}, ' + ('merge-aware' if mode else 'strict') if what == 'completeness' else None)
    ax.set_xticks(x, [SHORT.get(m, m) for m in methods], fontsize=8)
    ax.set_ylim(0, 1.02)
    ax.set_title(what + (f' (share of the λ = {REFERENCE} voids recovered)' if what == 'completeness'
                         else ' (share of the sparse-sample voids that are real)'), fontsize=10)
    ax.grid(alpha=0.3, axis='y')
axes[0].set_ylabel(f'share of voids matched (R_eff > {R_CUT:g} h⁻¹ Mpc)')
fig.legend(loc='upper center', bbox_to_anchor=(0.5, 0.0), ncol=2 * len(shown), fontsize=9, frameon=False)
fig.suptitle(f'{BOX}: each method at sparser spacings against itself at λ = {REFERENCE} '
             f'(last group: DeepVoid against the T-web truth, for scale)')
fig.tight_layout(rect=(0, 0, 1, 0.95))
fig.savefig(OUT / f'{BOX}_stability.png', dpi=150, bbox_inches='tight')
plt.show()

## Shut down

Unmounts Drive and deletes the runtime so it stops using compute units. Skip it if you want to keep working.

In [ ]:
from google.colab import runtime
drive.flush_and_unmount()
runtime.unassign()